# EECS 182, Fall 2026 - ReLU Networks with Different Optimizers

Run all active cells from top to bottom. Leave the optional from-scratch training block commented out: the supplied checkpoints contain everything needed for the questions.

We compare a one-hidden-layer ReLU network at widths 10, 20, and 40, trained with SGD, SGD with momentum, and Adam. An **elbow** is where a hidden ReLU changes between zero and a line. For a unit $\max(0, ax+b)$ with $a\ne0$, its elbow is $x=-b/a$.

Use the test-loss plots to compare typical errors and variation across 30 random seeds. Then compare the first and last saved elbow plots for one example seed. These are observations for the supplied data and learning rates, not guarantees about every optimization problem.

Use these plots to answer the accompanying handout questions. No new training or optimizer implementation is required. The Bug Hunt snippet is for reading only.

Contributors: Matteo Guarrera, Mert Cemri, Sizhe Chen, Kevin Li, Kumar Krishna Agrawal, Naman Jain.


In [ ]:
from pathlib import Path
import hashlib
import os
import subprocess
import sys
import urllib.request

# Use the reviewed helper and the original pretrained data at fixed revisions.
ASSET_COMMIT = "14fadfa21ba84c099322f4477cd0dfd50270da88"
HELPER_COMMIT = "1b155b9f340cf00a43b7bd6c57f1b2e91d1d14f2"
HELPER_SHA256 = "5e988680881671bbd0f0aef4454082366efb8cc565d250b087788ab3367ddaa4"
workspace = Path.cwd() / ".relu_optim_assets"
workspace.mkdir(exist_ok=True)
repository = workspace / ("checkpoints_" + ASSET_COMMIT[:12])
CKPT_ROOT = repository / "dis02/code/ckpts"

if not ((CKPT_ROOT / "tensors.pth").is_file()
        and (CKPT_ROOT / "nets_by_size.pth").is_file()
        and len(list(CKPT_ROOT.glob("*/width*/seed*/ckpt_and_history.pt"))) == 270):
    print("Loading pretrained checkpoints (no training required)...")
    repository.mkdir(exist_ok=True)
    def git(*args):
        subprocess.run(["git", "-C", str(repository), *args],
                       check=True, timeout=180)
    if not (repository / ".git").exists():
        git("init", "--quiet")
        git("remote", "add", "origin",
            "https://github.com/Berkeley-CS182/cs182fa26_public.git")
    git("fetch", "--quiet", "--depth", "1", "--filter=blob:none",
        "origin", ASSET_COMMIT)
    git("sparse-checkout", "set", "dis02/code/ckpts")
    git("checkout", "--quiet", "--detach", ASSET_COMMIT)

helper_path = workspace / "relu_optim_helpers.py"
if not (helper_path.is_file()
        and hashlib.sha256(helper_path.read_bytes()).hexdigest() == HELPER_SHA256):
    url = ("https://raw.githubusercontent.com/Berkeley-CS182/cs182fa26_public/"
           + HELPER_COMMIT + "/hw02/code/relu_optim_helpers.py")
    with urllib.request.urlopen(url, timeout=60) as response:
        helper_code = response.read()
    if hashlib.sha256(helper_code).hexdigest() != HELPER_SHA256:
        raise RuntimeError("The plotting helper does not match the reviewed version.")
    helper_path.write_bytes(helper_code)

sys.path.insert(0, str(workspace.resolve()))
sys.modules.pop("relu_optim_helpers", None)
print("Ready: supplied checkpoints and corrected plotting helper loaded.")


In [ ]:
# %load_ext autoreload
# %autoreload 2

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import copy
import time
import os
import sys
from ipywidgets import fixed, interactive, widgets 
from tqdm import tqdm

from relu_optim_helpers import *

%matplotlib inline

# Training and test data

We fit a piecewise linear target. Training labels contain Gaussian noise, $y=f(x)+\epsilon$, with $\epsilon\sim\mathcal N(0,\sigma^2)$. Test labels are noise free. The checkpoint-loading cell below restores the exact data used to train the supplied models.


In [ ]:
f_type = 'piecewise_linear'

def f_true(X, f_type):
    if f_type == 'sin(20x)':
        return np.sin(20 * X[:,0])
    else:
        TenX = 10 * X[:,0]
        _ = 12345
        return (TenX - np.floor(TenX)) * np.sin(_ * np.ceil(TenX)) - (TenX - np.ceil(TenX)) * np.sin(_ * np.floor(TenX)) 
    
n_features = 1
n_samples = 200
sigma = 0.01
rng = np.random.RandomState(1)

# Generate train data
X = np.sort(rng.rand(n_samples, n_features), axis=0)
y = f_true(X, f_type) + rng.randn(n_samples) * sigma

# Generate NOISELESS test data
X_test = np.concatenate([X.copy(), np.expand_dims(np.linspace(0., 1., 1000), axis=1)])
X_test = np.sort(X_test, axis=0)
y_test = f_true(X_test, f_type)

# Save checkpoint files
DIR_SGD = str(CKPT_ROOT / 'sgd')
DIR_SGDM = str(CKPT_ROOT / 'sgd_momentum')
DIR_ADAM = str(CKPT_ROOT / 'adam')
os.makedirs(DIR_SGD, exist_ok=True)
os.makedirs(DIR_SGDM, exist_ok=True)
os.makedirs(DIR_ADAM, exist_ok=True)

def get_ckpt_dir(optim: str):
    if optim == 'sgd':
        return DIR_SGD
    elif optim == 'sgd_momentum':
        return DIR_SGDM
    elif optim == 'adam':
        return DIR_ADAM
    else:
        raise NotImplementedError


# Define the networks

The one-hidden-layer model is
$$\hat y = W^{(2)}\operatorname{ReLU}(W^{(1)}x+b^{(1)})+b^{(2)}.$$
The superscripts identify the layers. Hidden-layer width is the number of ReLU units. We train all parameters with SGD, SGD with momentum, or Adam. The supplied experiment uses learning rate 0.02 for each optimizer and momentum 0.9 for SGD with momentum.

The next cell defines the models for optional training; the checkpoint-loading cell restores the supplied trained models.


In [ ]:
# Don't rerun this cell after training or you will lose all your work
nets_by_size = {}
nn_widths = [10, 20, 40]
nn_optimizer = ['sgd', "sgd_momentum", 'adam']
nn_seeds = [
  442, 370, 378, 892, 836, 209, 327, 316, 216, 308,
  748, 934, 558, 546, 266, 808, 884, 818, 277, 979, 
  766, 274, 479, 325, 431, 971, 689, 871, 272, 704
]

def setup_networks(widths, optimizer, seed):
  
  torch.manual_seed(seed)
  nets_by_size[seed] = dict()

  for width in widths:

      nets_by_size[seed][width] = dict()
      # Define a 1-hidden layer ReLU nonlinearity network. 
      # Initialize outside the optimizer loop to keep weight init the same.
      net = nn.Sequential(
        nn.Linear(1, width),
        nn.ReLU(),
        nn.Linear(width, 1)
      )

      for optim in optimizer:
        
        # Clone the network
        network = copy.deepcopy(net)

        # Get trainable parameters
        weights_all = list(network.parameters())
        
        # Get the output weights alone
        weights_out = weights_all[2:]
        # Adjust initial biases so elbows are in [0,1]
        elbows = np.sort(np.random.rand(width)) 
        new_biases = -elbows * to_numpy(weights_all[0].cpu()).ravel()
        weights_all[1].data = to_torch(new_biases)
        # Create SGD optimizers for outputs alone and for all weights
        # lr_out = 0.2
        lr_all = 0.02
        if optim == 'sgd':
          opt_all = torch.optim.SGD(params=weights_all, lr=lr_all)
        elif optim == 'sgd_momentum':
          opt_all = torch.optim.SGD(params=weights_all, lr=lr_all, momentum=0.9)
        elif optim == 'adam':
          opt_all = torch.optim.Adam(params=weights_all, lr=lr_all )
        # opt_out = torch.optim.SGD(params=weights_out, lr=lr_out)
        nets_by_size[seed][width][optim] = {
          'net': network,
          'opt_all': opt_all,
          'optim': optim,
          'seed': seed
        }

for s in nn_seeds:
  setup_networks(nn_widths, nn_optimizer, s)

# Optional: train from scratch

Leave the next cell commented out for this exercise. It would retrain 270 networks for 30,000 updates each; training time depends on the machine. Use the supplied checkpoints instead.


In [ ]:
# n_steps = 30000
# save_every = 3000 #1000
# t0 = time.time()

# def train_all_seeds(widths, optims, seeds):
  
#     for w in widths:
#       for i, optim in enumerate(optims):

#         print("-"*40)
#         print("Width", w, "Optimizer", optim)
#         list_of_history = []

#         print(f"training with {len(seeds)} seeds...")
#         for seed in tqdm(seeds):
#           net = nets_by_size[seed][w][optim]['net']
#           opt_all = nets_by_size[seed][w][optim]['opt_all']

#           save_dir = f'{get_ckpt_dir(optim)}/width{w}/seed{seed}/'
#           os.makedirs(save_dir, exist_ok=True)

#           history_all = train_network(X, y, X_test, y_test, 
#                                   net, optim=opt_all, 
#                                   n_steps=n_steps, save_every=save_every, 
#                                   verbose=False, optimizer=optim, seed=seed,
#                                   ckpt_dir=save_dir)
#           nets_by_size[seed][w][optim]['hist_all'] = history_all
#           list_of_history.append(history_all)  
      
# train_all_seeds(widths=nn_widths, optims=nn_optimizer, seeds=nn_seeds)
  
# t1 = time.time()
# print("-"*40)
# print("Trained all layers in %.1f minutes" % ((t1 - t0) / 60))

# # Compile the tensors into a dictionary
# data_dict = {
#     'X': X,
#     'y': y,
#     'X_test': X_test,
#     'y_test': y_test
# }

# torch.save(data_dict, "ckpts/tensors.pth")
# torch.save(nets_by_size, "ckpts/nets_by_size.pth")

The code cell below loads pretrained checkpoints for the models trained with SGD, SGD with momentum, and Adam.

In [ ]:
loaded_data_dict = torch.load(CKPT_ROOT / "tensors.pth", map_location="cpu", weights_only=False)
X = loaded_data_dict['X']
y = loaded_data_dict['y']
X_test = loaded_data_dict['X_test']
y_test = loaded_data_dict['y_test']
del loaded_data_dict

nets_by_size = torch.load(CKPT_ROOT / "nets_by_size.pth", map_location="cpu", weights_only=False)

## Plot Training Losses

In [ ]:
for w in nn_widths:
    fig, ax = plt.subplots(figsize=(12, 8))
    for i, optim in enumerate(nn_optimizer):
        
        # uncomment below if you ran the previous block to train all models
        # list_of_history = [nets_by_size[s][w][optim]['hist_all'] for s in nn_seeds]
        
        # uncomment below if you want to use pretrained weights
        c = get_ckpt_dir(optim)
        list_of_history = [
            torch.load(
                os.path.join(f'{get_ckpt_dir(optim)}/width{w}/seed{s}', 'ckpt_and_history.pt'),
                map_location='cpu', weights_only=True
            ) for s in nn_seeds
        ]
        plot_with_error_bar(list_of_history, optim=optim, plot_train=True, idx=i, ax=ax)
    ax.set_title(f"width {w}")
    plt.show()

## Plot test losses

Dots show the median mean-squared error across 30 random seeds. Error bars run from the 25th to the 75th percentile (the middle half of the runs); they are not confidence intervals. The vertical axis is logarithmic. Use these plots for part (a).


In [ ]:
for w in nn_widths:
    fig, ax = plt.subplots(figsize=(12, 8))
    for i, optim in enumerate(nn_optimizer):

        # uncomment below if you ran the previous block to train all models
        # list_of_history = [nets_by_size[s][w][optim]['hist_all'] for s in nn_seeds]
        
        # uncomment below if you want to use pretrained weights
        c = get_ckpt_dir(optim)
        list_of_history = [
            torch.load(
                os.path.join(f'{get_ckpt_dir(optim)}/width{w}/seed{s}', 'ckpt_and_history.pt'),
                map_location='cpu', weights_only=True
            ) for s in nn_seeds
        ]
        plot_with_error_bar(list_of_history, optim=optim, plot_test=True, idx=i, ax=ax)  
    ax.set_title(f"width {w}")
    plt.show()

## Compare elbow positions near the start and at the end

For one example seed, compare the first saved checkpoint (after one update) with the final checkpoint. Gray dots mark hidden-unit elbows, the green curve is the learned function, and the dashed blue curve is the target. Use these views for part (b). Other seeds can produce different elbow arrangements.


In [ ]:
SAMPLE = nn_seeds[0]
for w in nn_widths:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
    for col, optim in enumerate(nn_optimizer):
        history = torch.load(
            Path(get_ckpt_dir(optim)) / f'width{w}/seed{SAMPLE}/ckpt_and_history.pt',
            map_location='cpu', weights_only=True)
        net = copy.deepcopy(nets_by_size[SAMPLE][w][optim]['net']).cpu()
        for row, step in enumerate([min(history), max(history)]):
            plot_update(X, y, X_test, y_test, net,
                        state=history[step]['state'], optim=optim, ax=axes[row, col])
            axes[row, col].set_title(f'{optim}, step {step}')
        # Keep the final comparison grid consistent with these saved histories.
        nets_by_size[SAMPLE][w][optim]['net'].load_state_dict(history[max(history)]['state'])
    fig.suptitle(f'Width {w}, example seed {SAMPLE}: first and last saved checkpoints')
    plt.tight_layout()
    plt.show()


## TODO: Bug Hunt - Accumulating Gradients

The training loop below is shown for inspection only. It is a Markdown code block, not an executable cell, so **Run all will not execute the bug**.

```python
model.train()
optimizer.zero_grad()

for inputs, labels in training_loader:
    predictions = model(inputs)
    loss = loss_fn(predictions, labels)
    loss.backward()
    optimizer.step()
```

**TODO:**
1. Identify the bug and explain why it can make training unstable.
2. If the first two batch gradients are $g_1$ and $g_2$, what is stored in each parameter's `.grad` field after the second `loss.backward()` call?
3. Does `optimizer.step()` clear `.grad`?
4. Write or state the corrected per-batch operation order. You do not need to edit or execute this snippet.


## Questions

- How do hidden-layer width and optimizer choice affect the learned function and test error in these experiments? Compare the medians and variation across runs.
- Compare the first and last saved checkpoints for the example run. How do the elbow locations change relative to the target's kinks under each optimizer?
- Answer the Bug Hunt questions above.


In [ ]:
# Create a single figure with a 3x3 grid of subplots
fig, axs = plt.subplots(3, 3, figsize=(15, 15))

# Flatten the axs array for easy indexing
axs = axs.ravel()

# Iterate through the widths and optimizers and plot in subplots
for i, w in enumerate(nn_widths):
    for j, optim in enumerate(nn_optimizer):
        ax = axs[i * 3 + j]
        net = nets_by_size[SAMPLE][w][optim]['net']
        plot_update(X, y, X_test, y_test, net, optim=optim, ax=ax)
        ax.set_title(f"width {w}, {optim}")

# Adjust subplot layout
plt.tight_layout()
plt.show()
